# 03 LightGBM LambdaRank

Notebook ini adalah tahap **M1**: model ranking utama LightGBM LambdaRank dibandingkan dengan baseline dari `02_baseline.ipynb`.

Protokol supaya hasil test jujur:

1. Model dilatih di split **train** (April 2025 s.d. Maret 2026).
2. Early stopping dan tuning kecil hanya memakai split **val** (April s.d. Juni 2026).
3. Split **test** (Juli s.d. September 2026) dievaluasi **satu kali** di bagian 4, setelah model disimpan. Sel-sel sesudahnya hanya menampilkan hasil dari evaluasi itu.

Tabel utama memakai **grup informatif**, yaitu grup yang punya minimal dua grade berbeda. Grup yang semua grade-nya sama selalu mendapat nilai sempurna untuk model apa pun, jadi tidak membedakan model.

In [1]:
import sys
from datetime import date
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"


def find_ml_dir() -> Path:
    # notebook bisa dijalankan dari ml/notebooks, ml, atau root repo
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / "ml"):
            if (candidate / "procurewise_ml" / "__init__.py").exists():
                return candidate
    raise FileNotFoundError("folder ml tidak ditemukan")


sys.path.insert(0, str(find_ml_dir()))

from procurewise_ml import config, ranker, stability
from procurewise_ml import evaluation as ev
from procurewise_ml.baselines import BASELINES
from procurewise_ml.data import load_dataset

dataset, spec = load_dataset()
assert spec["source"]["seed"] == 15037, "notebook ini disusun untuk data seed 15037"
baseline_scores = {name: fn(dataset) for name, fn in BASELINES.items()}
print(f"lightgbm {lgb.__version__}, seed data {spec['source']['seed']}, {spec['rows']:,} pasangan")


def fmt(table: pd.DataFrame) -> pd.DataFrame:
    out = table.copy()
    for column in out.columns:
        if column == "n_grup":
            out[column] = out[column].astype(int)
        elif column in ("on_time", "defect_rate", "top1_grade_0"):
            out[column] = out[column].astype(float).map("{:.1%}".format)
        elif column == "total_biaya":
            out[column] = (out[column].astype(float) / 1e6).map("Rp{:,.1f} jt".format)
        else:
            out[column] = out[column].astype(float).round(3)
    return out

lightgbm 4.7.0, seed data 15037, 9,754 pasangan


## 1. Konfirmasi Bobot Baseline

Bobot `weighted_sum`, `topsis`, dan `topsis_priority` ditulis di `procurewise_ml/config.py` **sebelum** evaluasi pertama dijalankan di tahap S5, dan tidak pernah diubah setelah melihat hasil split mana pun, termasuk test. Karena itu bobot tidak perlu disetel ulang. Sebagai catatan, metrik baseline di val ditampilkan di bawah ini.

In [2]:
baseline_val = ev.compare(dataset, baseline_scores, "val", informative_only=True)
fmt(baseline_val[["n_grup", "ndcg@3", "hit@1", "mrr"]])

,n_grup,ndcg@3,hit@1,mrr
weighted_sum,319,0.904,0.824,0.901
topsis,319,0.902,0.812,0.893
topsis_priority,319,0.905,0.796,0.890


## 2. Tuning Kecil di Split Val

Grid berisi 12 kombinasi `learning_rate`, `num_leaves`, dan `min_child_samples`. Tiap kombinasi dilatih di train dengan early stopping 100 ronde terhadap NDCG@3 val, lalu dinilai dengan NDCG@3 grup informatif val. Test tidak dipakai di bagian ini.

In [3]:
tuning, best_params = ranker.tune(dataset)
print("hyperparameter terpilih:", best_params)
tuning.round(4)

hyperparameter terpilih: {'learning_rate': 0.1, 'num_leaves': 31, 'min_child_samples': 40}


,learning_rate,num_leaves,min_child_samples,best_iteration,val_ndcg@3_informatif,val_hit@1_informatif,val_ndcg@3_lightgbm
0,0.10,31,40,30,0.9806,0.9655,0.9830
1,0.03,31,40,63,0.9804,0.9592,0.9828
2,0.10,7,10,129,0.9795,0.9624,0.9820
3,0.03,31,10,84,0.9790,0.9592,0.9816
4,0.03,7,40,445,0.9785,0.9561,0.9812
5,0.10,15,10,222,0.9780,0.9498,0.9807
6,0.03,15,40,112,0.9776,0.9530,0.9803
7,0.10,15,40,35,0.9774,0.9467,0.9802
8,0.10,31,10,155,0.9772,0.9498,0.9801
9,0.03,7,10,417,0.9772,0.9561,0.9801


**Temuan:** Kedua belas kombinasi menghasilkan NDCG@3 informatif val di rentang sempit 0,977 sampai 0,981, jadi hasil tidak bergantung pada satu kombinasi tertentu. Kombinasi terpilih (`learning_rate` 0,1, `num_leaves` 31, `min_child_samples` 40) berhenti di iterasi ke-30.

## 3. Model Final dan Penyimpanan

Model final dilatih ulang dengan hyperparameter terpilih, lalu disimpan ke `ml/models/lgbm_v1.txt` beserta metadata. Model yang dimuat ulang dari file dicek menghasilkan skor yang sama.

In [4]:
trained = ranker.train_ranker(dataset, best_params)
lgbm_scores = ranker.predict_scores(trained.model, dataset)
val_scores = {"lightgbm": lgbm_scores, **baseline_scores}
val_table = ev.compare(dataset, val_scores, "val", informative_only=True)

metadata = {
    "model_version": config.MODEL_VERSION,
    "created": date.today().isoformat(),
    "lightgbm_version": lgb.__version__,
    "objective": "lambdarank",
    "params": {**config.LGBM_BASE_PARAMS, **best_params},
    "best_iteration": trained.best_iteration,
    "feature_columns": list(config.FEATURE_COLUMNS),
    "label": "grade 0 sampai 3 dari _labels.csv",
    "data": {**spec["source"], "train": "fit", "val": "early stopping dan tuning", "test": "evaluasi akhir sekali"},
    "tuning_grid": {k: list(v) for k, v in config.LGBM_TUNING_GRID.items()},
    "metrics": {"val_informatif": val_table.loc["lightgbm", ["n_grup", "ndcg@3", "hit@1", "mrr"]].astype(float).to_dict()},
}
model_path, meta_path = ranker.save_model(trained, metadata)

booster, _ = ranker.load_model()
reloaded = ranker.predict_scores(booster, dataset)
assert np.allclose(reloaded.dropna(), lgbm_scores.dropna()), "model dari file berbeda"
print(f"tersimpan: {model_path.name} ({model_path.stat().st_size / 1024:.0f} KB), {meta_path.name}")
print(f"best_iteration {trained.best_iteration}, model dari file menghasilkan skor yang sama")
fmt(val_table[["n_grup", "ndcg@3", "hit@1", "mrr"]])

tersimpan: lgbm_v1.txt (103 KB), lgbm_v1.json
best_iteration 30, model dari file menghasilkan skor yang sama


,n_grup,ndcg@3,hit@1,mrr
lightgbm,319,0.981,0.966,0.982
weighted_sum,319,0.904,0.824,0.901
topsis,319,0.902,0.812,0.893
topsis_priority,319,0.905,0.796,0.890


**Temuan:** Di val LightGBM jauh di atas baseline (NDCG@3 0,981 berbanding 0,904 untuk weighted_sum). Angka ini optimistis karena val dipakai untuk early stopping dan tuning, sehingga perbandingan yang adil ada di test.

## 4. Evaluasi Test (satu kali)

Semua angka test di notebook ini dihitung di sel berikut dari skor model yang sudah disimpan di atas. Metrik bisnis dihitung untuk vendor peringkat 1 memakai hasil simulasi dari label (`on_time` terhadap `needed_by`, perkiraan `defect_rate`, biaya efektif).

In [5]:
scores = {"lightgbm": lgbm_scores, **baseline_scores, **ev.reference_scores(dataset)}
test_informative = ev.compare(dataset, scores, "test", informative_only=True)
test_all = ev.compare(dataset, scores, "test", informative_only=False)
test_segments = ev.segment_compare(dataset, {"lightgbm": lgbm_scores, "weighted_sum": baseline_scores["weighted_sum"]},
                                   "test", informative_only=True)
test_segments_all = ev.segment_compare(dataset, {"lightgbm": lgbm_scores, "weighted_sum": baseline_scores["weighted_sum"]},
                                       "test", informative_only=False)
bootstrap = pd.concat({
    "informatif": ev.paired_bootstrap(dataset, lgbm_scores, baseline_scores["weighted_sum"], "test", informative_only=True),
    "semua grup": ev.paired_bootstrap(dataset, lgbm_scores, baseline_scores["weighted_sum"], "test", informative_only=False),
})

metadata["metrics"]["test_informatif"] = test_informative.loc["lightgbm", ["n_grup", "ndcg@3", "hit@1", "mrr"]].astype(float).to_dict()
metadata["metrics"]["test_semua"] = test_all.loc["lightgbm", ["n_grup", "ndcg@3", "hit@1", "mrr"]].astype(float).to_dict()
metadata["metrics"]["test_weighted_sum_informatif"] = test_informative.loc["weighted_sum", ["ndcg@3", "hit@1", "mrr"]].astype(float).to_dict()
metadata["metrics"]["bootstrap_vs_weighted_sum"] = {
    f"{group} {metric}": row[["selisih", "ci_95_bawah", "ci_95_atas"]].astype(float).to_dict()
    for (group, metric), row in bootstrap.iterrows()
}
ranker.save_model(trained, metadata)

print("tabel utama: grup informatif split test")
fmt(test_informative)

tabel utama: grup informatif split test


,n_grup,ndcg@3,hit@1,mrr,on_time,defect_rate,total_biaya,biaya_relatif,rata_grade,top1_grade_0
lightgbm,194,0.961,0.881,0.934,91.7%,1.3%,Rp363.1 jt,1.064,2.555,1.4%
weighted_sum,194,0.930,0.835,0.908,90.9%,1.5%,Rp356.7 jt,1.042,2.500,2.8%
topsis,194,0.901,0.773,0.871,89.8%,1.2%,Rp379.0 jt,1.116,2.440,3.2%
topsis_priority,194,0.925,0.804,0.897,90.7%,1.3%,Rp379.0 jt,1.099,2.482,1.8%
acak,194,0.750,0.392,0.634,85.2%,2.3%,Rp387.1 jt,1.158,1.904,7.8%
pilihan_historis,194,0.736,0.433,0.619,87.3%,3.1%,Rp352.4 jt,1.039,1.991,6.9%
oracle,194,1.000,1.000,1.000,94.1%,1.3%,Rp365.2 jt,1.074,2.670,0.0%


**Temuan:** LightGBM mengalahkan weighted_sum di grup informatif test, dengan NDCG@3 0,961 berbanding 0,930 dan Hit@1 0,881 berbanding 0,835. Vendor peringkat 1-nya lebih sering tepat waktu (91,7% berbanding 90,9%) dan lebih jarang bergrade 0 (1,4% berbanding 2,8%), tapi sedikit lebih mahal (biaya relatif 1,064 berbanding 1,042 kali kandidat termurah).

In [6]:
print("semua grup ranking split test")
fmt(test_all)

semua grup ranking split test


,n_grup,ndcg@3,hit@1,mrr,on_time,defect_rate,total_biaya,biaya_relatif,rata_grade,top1_grade_0
lightgbm,218,0.965,0.894,0.941,91.7%,1.3%,Rp363.1 jt,1.064,2.555,1.4%
weighted_sum,218,0.937,0.853,0.918,90.9%,1.5%,Rp356.7 jt,1.042,2.500,2.8%
topsis,218,0.912,0.798,0.885,89.8%,1.2%,Rp379.0 jt,1.116,2.440,3.2%
topsis_priority,218,0.933,0.826,0.908,90.7%,1.3%,Rp379.0 jt,1.099,2.482,1.8%
acak,218,0.778,0.459,0.674,85.2%,2.3%,Rp387.1 jt,1.158,1.904,7.8%
pilihan_historis,218,0.765,0.495,0.661,87.3%,3.1%,Rp352.4 jt,1.039,1.991,6.9%
oracle,218,1.000,1.000,1.000,94.1%,1.3%,Rp365.2 jt,1.074,2.670,0.0%


**Temuan:** Di semua 218 grup arahnya sama (NDCG@3 0,965 berbanding 0,937 dan Hit@1 0,894 berbanding 0,853). Turunnya NDCG@3 dari 0,981 di val ke 0,961 di test menunjukkan sebagian keunggulan di val memang efek tuning, tapi keunggulan atas baseline tetap ada di data yang belum pernah dilihat model.

## 5. Breakdown per Segmen (test)

Segmen dihitung per request dari fitur yang terlihat:

- **priority**: high, normal, low
- **slack**: `ketat` kalau median `slack_days` kandidat lolos <= 0 hari, artinya janji lead time vendor yang tipikal sudah menghabiskan seluruh waktu sampai `needed_by`. Sisanya `longgar`
- **musim**: `puncak` kalau `is_peak_season = 1`, selain itu `normal`
- **kategori material**: bahan, jasa, packaging

In [7]:
def segment_view(table: pd.DataFrame) -> pd.DataFrame:
    out = table.copy()
    out["selisih_ndcg@3"] = out["ndcg@3_lightgbm"] - out["ndcg@3_weighted_sum"]
    out["selisih_hit@1"] = out["hit@1_lightgbm"] - out["hit@1_weighted_sum"]
    columns = ["n_grup", "ndcg@3_lightgbm", "ndcg@3_weighted_sum", "selisih_ndcg@3",
               "hit@1_lightgbm", "hit@1_weighted_sum", "selisih_hit@1"]
    return out[columns].astype({"n_grup": int}).round(3)


print("grup informatif split test")
segment_view(test_segments)

grup informatif split test


,n_grup,ndcg@3_lightgbm,ndcg@3_weighted_sum,selisih_ndcg@3,hit@1_lightgbm,hit@1_weighted_sum,selisih_hit@1
segmen,,,,,,,
material_category=bahan,98,0.970,0.939,0.031,0.918,0.837,0.082
material_category=jasa,62,0.969,0.908,0.061,0.919,0.839,0.081
material_category=packaging,34,0.918,0.942,-0.024,0.706,0.824,-0.118
musim=normal,194,0.961,0.930,0.031,0.881,0.835,0.046
priority=high,59,0.952,0.923,0.029,0.797,0.746,0.051
priority=low,45,0.979,0.953,0.026,0.956,0.956,0.000
priority=normal,90,0.956,0.922,0.035,0.900,0.833,0.067
slack=ketat,52,0.934,0.898,0.036,0.788,0.673,0.115
slack=longgar,142,0.970,0.941,0.029,0.915,0.894,0.021


In [8]:
print("semua grup ranking split test")
segment_view(test_segments_all)

semua grup ranking split test


,n_grup,ndcg@3_lightgbm,ndcg@3_weighted_sum,selisih_ndcg@3,hit@1_lightgbm,hit@1_weighted_sum,selisih_hit@1
segmen,,,,,,,
material_category=bahan,102,0.971,0.941,0.030,0.922,0.843,0.078
material_category=jasa,65,0.971,0.912,0.058,0.923,0.846,0.077
material_category=packaging,51,0.946,0.961,-0.016,0.804,0.882,-0.078
musim=normal,218,0.965,0.937,0.028,0.894,0.853,0.041
priority=high,59,0.952,0.923,0.029,0.797,0.746,0.051
priority=low,57,0.984,0.963,0.021,0.965,0.965,0.000
priority=normal,102,0.962,0.931,0.030,0.912,0.853,0.059
slack=ketat,53,0.935,0.900,0.035,0.792,0.679,0.113
slack=longgar,165,0.974,0.949,0.025,0.927,0.909,0.018


**Temuan:** LightGBM paling unggul di request slack ketat (Hit@1 0,788 berbanding 0,673) dan kategori jasa (NDCG@3 +0,061), yaitu situasi saat janji lead time dan histori vendor harus dibaca bersama. Satu-satunya segmen yang kalah adalah packaging (NDCG@3 0,918 berbanding 0,942 dan Hit@1 0,706 berbanding 0,824 di 34 grup informatif), sedangkan di priority low Hit@1 keduanya sama.

### Musim Puncak vs Normal

Split test (Juli s.d. September 2026) tidak berisi satu pun request di musim ramai, jadi perbandingan musim puncak vs normal **tidak bisa** dihitung di test. Sebagai gambaran, tabel di bawah memakai split val. Angka val cenderung optimistis untuk LightGBM karena val dipakai untuk early stopping dan tuning.

In [9]:
val_segments = ev.segment_compare(dataset, {"lightgbm": lgbm_scores, "weighted_sum": baseline_scores["weighted_sum"]},
                                  "val", informative_only=True)
segment_view(val_segments.loc[[i for i in val_segments.index if i.startswith("musim")]])

,n_grup,ndcg@3_lightgbm,ndcg@3_weighted_sum,selisih_ndcg@3,hit@1_lightgbm,hit@1_weighted_sum,selisih_hit@1
segmen,,,,,,,
musim=normal,32,0.991,0.920,0.071,1.000,0.875,0.125
musim=puncak,287,0.979,0.902,0.077,0.962,0.819,0.143


**Temuan:** Di val, LightGBM unggul baik di musim puncak (287 grup, NDCG@3 0,979 berbanding 0,902) maupun normal (32 grup). Karena test tidak punya musim ramai dan val dipakai untuk tuning, kemampuan model saat musim ramai belum teruji secara bersih dan perlu uji skenario terpisah.

## 6. Bootstrap 95% CI Selisih terhadap Weighted Sum (test)

Selisih = metrik LightGBM dikurangi metrik weighted_sum di grup yang sama. CI dihitung dengan bootstrap berpasangan per grup sebanyak 2000 kali (seed 0). `p_selisih_le_0` adalah proporsi sampel bootstrap dengan selisih <= 0.

In [10]:
bootstrap.astype(float).round(4)

n_grup  selisih  ci_95_bawah  ci_95_atas  p_selisih_le_0
informatif ndcg@3   194.0   0.0310       0.0139      0.0468          0.0000
           hit@1    194.0   0.0464       0.0000      0.0979          0.0400
semua grup ndcg@3   218.0   0.0276       0.0127      0.0414          0.0005
           hit@1    218.0   0.0413       0.0000      0.0826          0.0330

**Temuan:** Selisih NDCG@3 signifikan, dengan CI 95% di grup informatif +0,014 sampai +0,047 yang tidak memuat 0. Selisih Hit@1 (+0,046) belum bisa disebut signifikan karena batas bawah CI tepat di 0, dan sekitar 4% sampel bootstrap menunjukkan LightGBM tidak lebih baik.

## 7. Kestabilan di Seed Lain

Pipeline yang sama (build dataset, training di train dengan early stopping di val, evaluasi test) dijalankan di data seed 1, 2, 3, 42, dan 43 dengan hyperparameter yang sama persis dengan model di atas. Tidak ada tuning ulang. Data tiap seed dibuat oleh generator dan disimpan di `ml/data/seeds/`.

In [11]:
stable = stability.run_stability(best_params)
columns = ["best_iteration", "n_grup_informatif",
           "ndcg@3_lgbm_informatif", "ndcg@3_ws_informatif", "ndcg@3_selisih_informatif",
           "hit@1_lgbm_informatif", "hit@1_ws_informatif", "hit@1_selisih_informatif",
           "ndcg@3_selisih_semua", "hit@1_selisih_semua"]
summary = stable[columns].copy()
summary.loc["rata-rata"] = summary.mean()
summary.loc["std"] = stable[columns].std()
print("lightgbm unggul ndcg@3 informatif di", int((stable["ndcg@3_selisih_informatif"] > 0).sum()), "dari", len(stable), "seed")
print("lightgbm unggul hit@1 informatif di", int((stable["hit@1_selisih_informatif"] > 0).sum()), "dari", len(stable), "seed")
summary.round(3)

lightgbm unggul ndcg@3 informatif di 5 dari 5 seed
lightgbm unggul hit@1 informatif di 4 dari 5 seed


,best_iteration,n_grup_informatif,ndcg@3_lgbm_informatif,ndcg@3_ws_informatif,ndcg@3_selisih_informatif,hit@1_lgbm_informatif,hit@1_ws_informatif,hit@1_selisih_informatif,ndcg@3_selisih_semua,hit@1_selisih_semua
seed,,,,,,,,,,
1,6.000,218.000,0.973,0.879,0.095,0.922,0.661,0.261,0.085,0.234
2,54.000,183.000,0.976,0.936,0.039,0.918,0.842,0.077,0.033,0.065
3,89.000,204.000,0.968,0.924,0.044,0.882,0.819,0.064,0.040,0.058
42,46.000,219.000,0.964,0.947,0.016,0.881,0.886,-0.005,0.015,-0.004
43,158.000,171.000,0.957,0.934,0.022,0.848,0.842,0.006,0.016,0.004
rata-rata,70.600,199.000,0.968,0.924,0.043,0.890,0.810,0.081,0.038,0.071
std,57.077,21.366,0.008,0.027,0.031,0.030,0.087,0.107,0.028,0.096


**Temuan:** Dengan hyperparameter yang sama, LightGBM unggul NDCG@3 di kelima seed (selisih +0,016 sampai +0,095, rata-rata +0,043) dan unggul Hit@1 di 4 dari 5 seed. Seed 42 adalah satu-satunya yang Hit@1-nya sedikit kalah (-0,005), sedangkan seed 1 memberi keunggulan terbesar karena baseline-nya paling lemah di seed itu.

## 8. Importance Fitur Global

Dua ukuran: **gain** (total perbaikan loss dari split yang memakai fitur itu) dan **rata-rata |SHAP|** dari `pred_contrib` di semua kandidat lolos pada grup ranking train, val, dan test.

In [12]:
ranked_rows = dataset[dataset["is_eligible"]]
ranked_rows = ranked_rows[ranked_rows.groupby("request_id")["vendor_id"].transform("size") >= config.MIN_ELIGIBLE_CANDIDATES]
contrib_all = ranker.contributions(booster, ranked_rows)
gain = pd.Series(booster.feature_importance("gain"), index=booster.feature_name())
importance = pd.DataFrame({
    "gain_persen": gain / gain.sum(),
    "mean_abs_shap": contrib_all[list(config.FEATURE_COLUMNS)].abs().mean(),
    "n_split": pd.Series(booster.feature_importance("split"), index=booster.feature_name()),
}).sort_values("mean_abs_shap", ascending=False)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Rata-rata |SHAP|", "Gain (persen)"), shared_yaxes=True)
order = importance.index[::-1]
fig.add_bar(y=order, x=importance.loc[order, "mean_abs_shap"], orientation="h", marker_color="#2a6fdb", row=1, col=1)
fig.add_bar(y=order, x=importance.loc[order, "gain_persen"], orientation="h", marker_color="#e07b39", row=1, col=2)
fig.update_xaxes(tickformat=".0%", row=1, col=2)
fig.update_layout(title="Importance fitur global lgbm-v1", height=520, showlegend=False)
fig.show()
importance.round(4)

,gain_persen,mean_abs_shap,n_split
avg_rating,0.3785,0.7454,90
slack_days,0.1578,0.3727,134
price_rel_median,0.1585,0.3668,119
qty_capacity_ratio,0.0668,0.2333,93
on_time_rate,0.0249,0.0978,46
unit_price,0.0471,0.0795,81
quoted_lead_time_days,0.0337,0.0763,48
defect_rate,0.0403,0.0675,71
n_transactions,0.0168,0.0569,38
total_cost,0.0165,0.0564,48


**Temuan:** Model paling banyak bergantung pada `avg_rating`, `slack_days`, dan `price_rel_median`, sejalan dengan tiga komponen label (kualitas, ketepatan waktu, dan biaya). `has_history` tidak dipakai sama sekali dan `is_peak_season` hanya sekali, kemungkinan karena informasinya sudah tercermin di `n_transactions`, `slack_days`, dan `qty_capacity_ratio`.

## 9. Contoh pred_contrib: Request Urgent dan Santai

Contoh diambil dari grup informatif di split test dengan minimal 3 kandidat lolos:

- **urgent**: priority `high` dan slack `ketat` (median `slack_days` <= 0)
- **santai**: priority `low` dan median `slack_days` >= 5 hari

Dari tiap jenis dipilih request dengan kandidat lolos terbanyak (seri dipecah dengan `request_id` terkecil). Kontribusi positif menaikkan skor vendor, negatif menurunkannya. Tiga kontribusi terbesar per vendor inilah yang nanti dikirim ke Gemini sebagai bahan penjelasan.

In [13]:
segments = ev.request_segments(dataset)
groups_test = ev.group_metrics(dataset, lgbm_scores)
groups_test = groups_test[(groups_test["split"] == "test") & groups_test["informative"]]
eligible_counts = dataset[dataset["is_eligible"]].groupby("request_id").size()
median_slack = dataset[dataset["is_eligible"]].groupby("request_id")["slack_days"].median()
pool = groups_test.join(segments[["priority", "slack"]]).assign(
    n_kandidat=eligible_counts, median_slack=median_slack)
pool = pool[pool["n_kandidat"] >= 3]


def pick(mask: pd.Series) -> int:
    chosen = pool[mask].sort_values(["n_kandidat"], ascending=False, kind="stable")
    chosen = chosen[chosen["n_kandidat"] == chosen["n_kandidat"].max()]
    return int(chosen.index.min())


examples = {
    "urgent": pick((pool["priority"] == "high") & (pool["slack"] == "ketat")),
    "santai": pick((pool["priority"] == "low") & (pool["median_slack"] >= 5)),
}
ranked_test = ev.rank_candidates(dataset, lgbm_scores)


def explain(request_id: int) -> pd.DataFrame:
    rows = ranked_test[ranked_test["request_id"] == request_id].sort_values("rank")
    contrib = ranker.contributions(booster, rows)[list(config.FEATURE_COLUMNS)]
    top3 = contrib.apply(lambda r: ", ".join(f"{f} {v:+.2f}" for f, v in r.reindex(r.abs().sort_values(ascending=False).index)[:3].items()), axis=1)
    return pd.DataFrame({
        "vendor_id": rows["vendor_id"],
        "rank_lgbm": rows["rank"],
        "skor": rows["score"].round(3),
        "grade": rows["grade"],
        "unit_price": rows["unit_price"],
        "slack_days": rows["slack_days"],
        "on_time_rate": rows["on_time_rate"].round(2),
        "defect_rate": rows["defect_rate"].round(3),
        "top3_kontribusi": top3,
    }).set_index("vendor_id"), contrib


for kind, request_id in examples.items():
    info = dataset[dataset["request_id"] == request_id].iloc[0]
    print(f"{kind}: request {request_id}, material {info['material_id']} ({info['material_category']}), "
          f"tanggal {info['request_date'].date()}, priority {segments.loc[request_id, 'priority']}, "
          f"median slack {median_slack[request_id]:.0f} hari")

urgent: request 1976, material 12 (packaging), tanggal 2026-08-15, priority high, median slack 0 hari
santai: request 1969, material 12 (packaging), tanggal 2026-08-13, priority low, median slack 12 hari


In [14]:
urgent_table, urgent_contrib = explain(examples["urgent"])
urgent_table

,rank_lgbm,skor,grade,unit_price,slack_days,on_time_rate,defect_rate,top3_kontribusi
vendor_id,,,,,,,,
21,1,0.874,2.0,4050.0,2.0,0.74,0.026,"price_rel_median +0.49, qty_capacity_ratio +0...."
28,2,0.755,2.0,4450.0,2.0,0.76,0.024,"qty_capacity_ratio +0.39, price_rel_median +0...."
27,3,0.533,3.0,5000.0,0.0,0.94,0.011,"avg_rating +0.36, slack_days +0.18, on_time_ra..."
20,4,0.439,2.0,4550.0,2.0,0.67,0.012,"slack_days +0.28, qty_capacity_ratio +0.28, av..."
23,5,-0.712,2.0,4150.0,1.0,0.63,0.049,"avg_rating -0.85, price_rel_median +0.38, slac..."
26,6,-1.432,0.0,5300.0,-5.0,0.78,0.007,"slack_days -1.60, avg_rating +0.93, price_rel_..."
24,7,-1.553,1.0,4700.0,-2.0,0.78,0.009,"slack_days -1.11, price_rel_median -0.19, on_t..."
25,8,-2.405,0.0,4750.0,-4.0,0.88,0.011,"slack_days -2.15, price_rel_median -0.17, qty_..."
22,9,-2.715,0.0,4750.0,-3.0,0.59,0.018,"slack_days -1.77, avg_rating -0.27, price_rel_..."


In [15]:
santai_table, santai_contrib = explain(examples["santai"])
santai_table

,rank_lgbm,skor,grade,unit_price,slack_days,on_time_rate,defect_rate,top3_kontribusi
vendor_id,,,,,,,,
27,1,0.993,2.0,5000.0,12.0,0.94,0.011,"avg_rating +0.59, slack_days +0.33, price_rel_..."
28,2,0.886,2.0,4450.0,14.0,0.76,0.024,"price_rel_median +0.37, slack_days +0.36, qty_..."
21,3,0.780,3.0,4050.0,14.0,0.74,0.026,"price_rel_median +0.53, qty_capacity_ratio +0...."
20,4,0.769,3.0,4550.0,14.0,0.67,0.012,"slack_days +0.44, avg_rating +0.31, qty_capaci..."
25,5,0.569,2.0,4750.0,8.0,0.88,0.011,"avg_rating +0.38, slack_days +0.34, qty_capaci..."
24,6,0.459,2.0,4700.0,10.0,0.78,0.009,"slack_days +0.41, avg_rating +0.38, price_rel_..."
22,7,-0.271,2.0,4750.0,9.0,0.59,0.018,"slack_days +0.37, price_rel_median -0.28, n_tr..."
23,8,-0.912,2.0,4150.0,13.0,0.63,0.049,"avg_rating -1.04, price_rel_median +0.35, slac..."


In [16]:
fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=(f"urgent, request {examples['urgent']}", f"santai, request {examples['santai']}"))
for col, contrib in ((1, urgent_contrib), (2, santai_contrib)):
    top_vendor = contrib.iloc[0]
    fig.add_bar(y=list(config.FEATURE_COLUMNS), x=top_vendor.to_numpy(), orientation="h",
                marker_color=np.where(top_vendor.to_numpy() >= 0, "#3c9d5d", "#d64545"), row=1, col=col)
fig.update_layout(title="Kontribusi fitur (pred_contrib) untuk vendor peringkat 1", height=560, showlegend=False)
fig.show()

**Temuan:** Kedua contoh kebetulan request kotak duplex (P02, packaging) dengan 8 sampai 9 kandidat, dan keduanya memperlihatkan kelemahan model di packaging karena vendor grade 3 hanya ditaruh di peringkat 3 (urgent) dan 3 sampai 4 (santai). Pola kontribusinya tetap masuk akal: di request urgent `slack_days` negatif menjatuhkan vendor 22, 24, 25, dan 26 ke bawah, sedangkan di request santai semua vendor punya slack lebar sehingga `avg_rating` dan harga yang menentukan urutan.

## Kesimpulan

1. **LightGBM (lgbm-v1) mengalahkan baseline terbaik weighted_sum di test.** NDCG@3 naik +0,031 dengan CI 95% +0,014 sampai +0,047, dan keunggulan NDCG@3 konsisten di 5 dari 5 seed lain.
2. **Hit@1 juga naik (+0,046) tapi belum signifikan** secara statistik di test seed 15037.
3. **Keunggulan terbesar ada di request slack ketat dan kategori jasa.** Model kalah dari weighted_sum di packaging, segmen yang kandidatnya paling banyak gugur filter dan vendornya cenderung mahal.
4. **Kemampuan saat musim ramai belum teruji bersih** karena test tidak memuat musim ramai.
5. **Model disimpan** di `ml/models/lgbm_v1.txt` dengan metadata di `ml/models/lgbm_v1.json` dan siap dimuat backend.